# CodonFM SAE — Linear Probing

Do individual SAE features encode interpretable biological properties? We test this by **linear probing**: for each candidate biological label (e.g., "is this a rare codon?"), we fit a simple logistic regression `sigmoid(w · z_i + b)` on every latent `z_i` independently and measure how well it predicts the label.

If a single latent achieves high AUROC for a label, that latent has learned a clean representation of that concept — it's a biological detector. If no single latent works but a linear combination does, the concept is distributed across features.

We implement two levels of probing:

**Codon-level probes** — one label per codon position:
- Rare codon detection (bottom quartile CAI for its amino acid)
- Start codon (ATG)
- GC-rich context (local GC content > 0.6)
- Wobble position GC (3rd nucleotide is G or C)

**Gene-level probes** — one label per gene (using mean-pooled activations):
- Housekeeping gene (constitutively expressed across tissues)
- High pLI (loss-of-function intolerant, pLI > 0.9)

For each probe, we report the best single latent, its AUROC, and the full distribution of per-latent AUROCs.

## Setup

In [ ]:
# ── Configure paths ──
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
NUM_SEQUENCES = 1000  # Use more for reliable results; 1000 is fast for iteration
DEVICE = "cuda"

# Optional: path to gnomAD pLI file for gene-level probing
PLI_PATH = None  # e.g., "../datasets/gnomad.v2.1.1.lof_metrics.by_gene.txt.bgz"

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load SAE, Model, and Data

In [ ]:
# Load SAE
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}
input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=model_config.get("top_k"),
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)
n_features = sae.hidden_dim
print(f"SAE: {input_dim} \u2192 {hidden_dim:,} features (top-{model_config.get('top_k')})")

# Load Encodon
inference = EncodonInference(model_path=MODEL_PATH, task_type="embedding_prediction", use_transformer_engine=True)
inference.configure_model()
inference.model.to(device).eval()
print(f"Encodon: {len(inference.model.model.layers)} layers")

# Load data
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)
sequences = [r.sequence for r in records]
gene_names = [r.metadata.get("gene", "") for r in records]
print(f"Loaded {len(sequences)} sequences")

## Extract SAE Activations

We stream sequences through the Encodon model and SAE encoder, collecting per-codon activations. For codon-level probes we need the full `(n_codons, n_features)` matrix; for gene-level probes we mean-pool across codons per gene.

To keep memory manageable, we process one sequence at a time and store only what we need: per-codon activations for codon-level probes, and running mean-pooled activations per gene for gene-level probes.

In [ ]:
# Collect per-codon SAE activations + metadata for probing
# We store: activations, codon strings, gene names, and sequence indices
# For memory, we cap total codons stored

MAX_CODONS = 500_000  # Cap for codon-level probes

all_activations = []  # list of numpy arrays, each (n_codons_in_seq, n_features)
all_codons = []  # list of codon string lists
all_gene_labels = []  # gene name per codon (repeated for each codon in gene)
total_codons = 0

# Gene-level: accumulate mean activations per gene
gene_act_sum = defaultdict(lambda: np.zeros(n_features, dtype=np.float64))
gene_act_count = defaultdict(int)

print(f"Extracting SAE activations (max {MAX_CODONS:,} codons)...")
with torch.no_grad():
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc="Extracting"):
        if total_codons >= MAX_CODONS:
            break
        batch_seqs = sequences[i : i + BATCH_SIZE]
        batch_genes = gene_names[i : i + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=inference.tokenizer) for s in batch_seqs]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER].float()
        attn = batch["attention_mask"]

        # Build keep mask (exclude CLS at 0, SEP at last position)
        keep = attn.clone()
        keep[:, 0] = 0
        lengths = attn.sum(dim=1)
        for b in range(keep.shape[0]):
            sep = int(lengths[b].item()) - 1
            if sep > 0:
                keep[b, sep] = 0

        for b in range(len(batch_seqs)):
            if total_codons >= MAX_CODONS:
                break
            vl = int(keep[b].sum().item())
            if vl == 0:
                continue
            emb = hidden[b, :vl, :]
            codes = sae.encode(emb).cpu().numpy()  # (vl, n_features)

            seq = batch_seqs[b]
            codons = [seq[j * 3 : (j + 1) * 3].upper() for j in range(vl)]
            gene = str(batch_genes[b]).strip() if batch_genes[b] else ""

            all_activations.append(codes)
            all_codons.append(codons)
            all_gene_labels.extend([gene] * vl)
            total_codons += vl

            # Gene-level accumulation
            if gene:
                gene_act_sum[gene] += codes.sum(axis=0)
                gene_act_count[gene] += vl

        del out, hidden, batch
        torch.cuda.empty_cache()

# Stack codon-level data
codon_acts = np.concatenate(all_activations, axis=0)  # (total_codons, n_features)
codon_strings = []
for cl in all_codons:
    codon_strings.extend(cl)

print(f"Collected {codon_acts.shape[0]:,} codons \u00d7 {codon_acts.shape[1]:,} features")
print(f"Gene-level: {len(gene_act_sum)} unique genes")

## Define Probe Labels

Each probe is a binary classification task. We construct labels from the codon sequences and metadata — no external data needed for the codon-level probes.

### Codon-Level Probes

| Probe | What it tests | Label = 1 when... |
|-------|---------------|-------------------|
| **Rare codon** | Does the feature detect translationally suboptimal codons? | CAI weight < 0.5 (bottom half for its amino acid) |
| **Start codon** | Does any feature specifically detect ATG? | Codon is ATG |
| **GC-rich wobble** | Does the feature track GC content at the wobble (3rd) position? | 3rd nucleotide is G or C |
| **CpG site** | Does the feature detect CpG dinucleotides across codon boundaries? | Last nt of this codon is C and first nt of next codon is G |

In [ ]:
# ── Codon optimality weights (same as notebook 02) ──
CODON_TO_AA = {
    "TTT": "F",
    "TTC": "F",
    "TTA": "L",
    "TTG": "L",
    "CTT": "L",
    "CTC": "L",
    "CTA": "L",
    "CTG": "L",
    "ATT": "I",
    "ATC": "I",
    "ATA": "I",
    "ATG": "M",
    "GTT": "V",
    "GTC": "V",
    "GTA": "V",
    "GTG": "V",
    "TCT": "S",
    "TCC": "S",
    "TCA": "S",
    "TCG": "S",
    "CCT": "P",
    "CCC": "P",
    "CCA": "P",
    "CCG": "P",
    "ACT": "T",
    "ACC": "T",
    "ACA": "T",
    "ACG": "T",
    "GCT": "A",
    "GCC": "A",
    "GCA": "A",
    "GCG": "A",
    "TAT": "Y",
    "TAC": "Y",
    "TAA": "*",
    "TAG": "*",
    "CAT": "H",
    "CAC": "H",
    "CAA": "Q",
    "CAG": "Q",
    "AAT": "N",
    "AAC": "N",
    "AAA": "K",
    "AAG": "K",
    "GAT": "D",
    "GAC": "D",
    "GAA": "E",
    "GAG": "E",
    "TGT": "C",
    "TGC": "C",
    "TGA": "*",
    "TGG": "W",
    "CGT": "R",
    "CGC": "R",
    "CGA": "R",
    "CGG": "R",
    "AGT": "S",
    "AGC": "S",
    "AGA": "R",
    "AGG": "R",
    "GGT": "G",
    "GGC": "G",
    "GGA": "G",
    "GGG": "G",
}

HUMAN_CODON_USAGE = {
    "TTT": 17.6,
    "TTC": 20.3,
    "TTA": 7.7,
    "TTG": 12.9,
    "CTT": 13.2,
    "CTC": 19.6,
    "CTA": 7.2,
    "CTG": 39.6,
    "ATT": 16.0,
    "ATC": 20.8,
    "ATA": 7.5,
    "ATG": 22.0,
    "GTT": 11.0,
    "GTC": 14.5,
    "GTA": 7.1,
    "GTG": 28.1,
    "TCT": 15.2,
    "TCC": 17.7,
    "TCA": 12.2,
    "TCG": 4.4,
    "CCT": 17.5,
    "CCC": 19.8,
    "CCA": 16.9,
    "CCG": 6.9,
    "ACT": 13.1,
    "ACC": 18.9,
    "ACA": 15.1,
    "ACG": 6.1,
    "GCT": 18.4,
    "GCC": 27.7,
    "GCA": 15.8,
    "GCG": 7.4,
    "TAT": 12.2,
    "TAC": 15.3,
    "TAA": 1.0,
    "TAG": 0.8,
    "CAT": 10.9,
    "CAC": 15.1,
    "CAA": 12.3,
    "CAG": 34.2,
    "AAT": 17.0,
    "AAC": 19.1,
    "AAA": 24.4,
    "AAG": 31.9,
    "GAT": 21.8,
    "GAC": 25.1,
    "GAA": 29.0,
    "GAG": 39.6,
    "TGT": 10.6,
    "TGC": 12.6,
    "TGA": 1.6,
    "TGG": 13.2,
    "CGT": 4.5,
    "CGC": 10.4,
    "CGA": 6.2,
    "CGG": 11.4,
    "AGT": 12.1,
    "AGC": 19.5,
    "AGA": 12.2,
    "AGG": 12.0,
    "GGT": 10.8,
    "GGC": 22.2,
    "GGA": 16.5,
    "GGG": 16.5,
}

# CAI weights: freq / max_freq for same amino acid
from collections import defaultdict as _dd


_aa_codons = _dd(list)
for c, aa in CODON_TO_AA.items():
    if aa != "*":
        _aa_codons[aa].append(c)

CAI_WEIGHTS = {}
for aa, codons in _aa_codons.items():
    freqs = [HUMAN_CODON_USAGE[c] for c in codons]
    mx = max(freqs)
    for c, f in zip(codons, freqs):
        CAI_WEIGHTS[c] = f / mx

# ── Build codon-level probe labels ──
n_codons_total = len(codon_strings)

probes = {}

# 1. Rare codon: CAI weight < 0.5
probes["rare_codon"] = np.array([1 if CAI_WEIGHTS.get(c, 1.0) < 0.5 else 0 for c in codon_strings], dtype=np.float32)

# 2. Start codon (ATG)
probes["start_codon_ATG"] = np.array([1 if c == "ATG" else 0 for c in codon_strings], dtype=np.float32)

# 3. GC-rich wobble position (3rd nt is G or C)
probes["wobble_GC"] = np.array(
    [1 if len(c) == 3 and c[2] in ("G", "C") else 0 for c in codon_strings], dtype=np.float32
)

# 4. CpG site (last nt = C, next codon first nt = G)
cpg_labels = np.zeros(n_codons_total, dtype=np.float32)
for j in range(n_codons_total - 1):
    c_curr = codon_strings[j]
    c_next = codon_strings[j + 1]
    # Only within same sequence (check gene label continuity as proxy)
    if len(c_curr) == 3 and len(c_next) >= 1 and all_gene_labels[j] == all_gene_labels[j + 1]:
        if c_curr[2] == "C" and c_next[0] == "G":
            cpg_labels[j] = 1.0
probes["CpG_site"] = cpg_labels

for name, labels in probes.items():
    n_pos = int(labels.sum())
    print(f"  {name}: {n_pos:,} positive / {len(labels):,} total ({n_pos / len(labels):.1%})")

## Single-Latent Probing

For each probe task and each SAE latent, we compute the AUROC of using that single latent's activation as a classifier score. This is the simplest possible probe — no learned parameters, just "does higher activation of latent `i` predict label = 1?"

We compute AUROC rather than accuracy because the labels are often imbalanced (e.g., only ~3% of codons are ATG). AUROC is threshold-independent and handles class imbalance naturally.

For efficiency, we compute all latent AUROCs in parallel using vectorized operations. With 32K features \u00d7 500K codons, this is the bottleneck — we subsample if needed.

In [ ]:
def compute_single_latent_aurocs(activations, labels, max_features=None, subsample=100_000):
    """Compute AUROC for each latent as a single-feature classifier.

    Args:
        activations: (n_samples, n_features) array
        labels: (n_samples,) binary array
        max_features: if set, only evaluate this many features (highest-variance)
        subsample: subsample to this many examples for speed

    Returns:
        aurocs: (n_features,) array of AUROC scores
        feature_indices: which features were evaluated (if max_features used)
    """
    n_samples, n_feats = activations.shape

    # Subsample for speed
    if subsample and n_samples > subsample:
        rng = np.random.RandomState(42)
        idx = rng.choice(n_samples, subsample, replace=False)
        activations = activations[idx]
        labels = labels[idx]
        n_samples = subsample

    # Filter to features that have any variance (dead features get AUROC=0.5)
    feat_var = activations.var(axis=0)
    if max_features and max_features < n_feats:
        feature_indices = np.argsort(feat_var)[-max_features:]
    else:
        feature_indices = np.where(feat_var > 0)[0]

    aurocs = np.full(n_feats, 0.5)

    pos_mask = labels == 1
    n_pos = pos_mask.sum()
    n_neg = n_samples - n_pos

    if n_pos < 5 or n_neg < 5:
        print(f"  WARNING: Too few positives ({n_pos}) or negatives ({n_neg}), skipping")
        return aurocs, feature_indices

    for feat_idx in tqdm(feature_indices, desc="  AUROC per latent", leave=False):
        try:
            aurocs[feat_idx] = roc_auc_score(labels, activations[:, feat_idx])
        except ValueError:
            pass  # constant predictions

    return aurocs, feature_indices


# Run codon-level probes
codon_probe_results = {}

for probe_name, labels in probes.items():
    print(f"\n{'=' * 50}")
    print(f"Probe: {probe_name}")
    aurocs, feat_idx = compute_single_latent_aurocs(codon_acts, labels, max_features=5000, subsample=100_000)

    best_idx = np.argmax(aurocs)
    best_auroc = aurocs[best_idx]
    top10 = np.argsort(aurocs)[-10:][::-1]

    print(f"  Best latent: {best_idx} (AUROC = {best_auroc:.4f})")
    print("  Top 10 latents:")
    for idx in top10:
        print(f"    Feature {idx:>5d}: AUROC = {aurocs[idx]:.4f}")

    codon_probe_results[probe_name] = {
        "aurocs": aurocs,
        "best_feature": int(best_idx),
        "best_auroc": float(best_auroc),
    }

## Visualize Probe Results

For each probe, we plot the distribution of per-latent AUROCs. A spike near 1.0 means one or more latents have learned an almost perfect detector for that concept. A distribution clustered around 0.5 means no single latent captures it — the concept is either not represented or distributed across multiple features.

In [ ]:
fig, axes = plt.subplots(1, len(probes), figsize=(5 * len(probes), 4))
if len(probes) == 1:
    axes = [axes]

colors = ["#76b900", "#0074DF", "#9525C6", "#EF2020"]

for ax, (probe_name, result), color in zip(axes, codon_probe_results.items(), colors):
    aurocs = result["aurocs"]
    alive_aurocs = aurocs[aurocs != 0.5]  # Exclude dead features

    ax.hist(alive_aurocs, bins=50, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(
        result["best_auroc"],
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Best: {result['best_auroc']:.3f} (#{result['best_feature']})",
    )
    ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("AUROC")
    ax.set_ylabel("Number of features")
    ax.set_title(probe_name.replace("_", " "))
    ax.legend(fontsize=9)
    ax.set_xlim(0.3, 1.0)

plt.tight_layout()
plt.show()

## Gene-Level Probes

For gene-level probing, we mean-pool SAE activations across all codons in each gene, giving one activation vector per gene. We then probe for gene-level properties.

We demonstrate with **pLI** (probability of loss-of-function intolerance): genes with pLI > 0.9 are essential and can't tolerate losing function. If an SAE feature has high AUROC for this label, it's detecting patterns specific to essential genes — potentially codon optimization signals, since essential genes tend to be highly expressed and use optimal codons.

In [ ]:
# Build gene-level mean activations
gene_names_unique = sorted(gene_act_sum.keys())
gene_mean_acts = np.array([gene_act_sum[g] / gene_act_count[g] for g in gene_names_unique])
print(f"Gene-level activations: {gene_mean_acts.shape[0]} genes \u00d7 {gene_mean_acts.shape[1]} features")

gene_probe_results = {}

# ── pLI probe (requires gnomAD file) ──
if PLI_PATH:
    import pandas as pd

    pli_df = pd.read_csv(PLI_PATH, sep="\t", compression="gzip", usecols=["gene", "pLI"]).dropna()
    pli_df = pli_df.drop_duplicates(subset=["gene"], keep="first")
    pli_map = dict(zip(pli_df["gene"], pli_df["pLI"]))

    pli_labels = np.array([1.0 if pli_map.get(g, 0) > 0.9 else 0.0 for g in gene_names_unique])
    n_constrained = int(pli_labels.sum())
    print(f"pLI probe: {n_constrained} constrained / {len(pli_labels)} genes ({n_constrained / len(pli_labels):.1%})")

    aurocs, feat_idx = compute_single_latent_aurocs(gene_mean_acts, pli_labels, max_features=5000)
    best_idx = np.argmax(aurocs)
    print(f"  Best latent: {best_idx} (AUROC = {aurocs[best_idx]:.4f})")
    gene_probe_results["pLI_constrained"] = {
        "aurocs": aurocs,
        "best_feature": int(best_idx),
        "best_auroc": float(aurocs[best_idx]),
    }
else:
    print("PLI_PATH not set \u2014 skipping pLI probe.")
    print("Set PLI_PATH to the gnomAD constraint file to enable gene-level pLI probing.")

# ── Housekeeping gene probe (heuristic: genes present in many tissues) ──
# As a simple proxy, genes with very common names (no tissue prefix) that appear
# frequently in the dataset are likely housekeeping. We use gene frequency as a proxy.
gene_freq = {g: gene_act_count[g] for g in gene_names_unique}
freq_values = np.array([gene_freq[g] for g in gene_names_unique])
# Top quartile by codon count = likely highly expressed / housekeeping
housekeeping_labels = (freq_values >= np.percentile(freq_values, 75)).astype(np.float32)
n_hk = int(housekeeping_labels.sum())
print(
    f"\nHousekeeping proxy probe: {n_hk} positive / {len(housekeeping_labels)} genes ({n_hk / len(housekeeping_labels):.1%})"
)

aurocs, feat_idx = compute_single_latent_aurocs(gene_mean_acts, housekeeping_labels, max_features=5000)
best_idx = np.argmax(aurocs)
print(f"  Best latent: {best_idx} (AUROC = {aurocs[best_idx]:.4f})")
gene_probe_results["housekeeping_proxy"] = {
    "aurocs": aurocs,
    "best_feature": int(best_idx),
    "best_auroc": float(aurocs[best_idx]),
}

## Logistic Regression Probe (Multi-Latent Baseline)

The single-latent AUROC tells us about monosemantic features. But how well can a *linear combination* of all latents predict each label? This is the upper bound for linear probing — if the best single latent gets AUROC 0.95 and the full logistic regression gets 0.96, the concept is cleanly captured by one feature. If the single best is 0.65 but the full model gets 0.95, the concept is distributed.

In [ ]:
from sklearn.model_selection import cross_val_score


print("Logistic regression probes (5-fold CV):\n")

# Codon-level (subsample for speed)
rng = np.random.RandomState(42)
n_sub = min(50_000, codon_acts.shape[0])
sub_idx = rng.choice(codon_acts.shape[0], n_sub, replace=False)
X_sub = codon_acts[sub_idx]

for probe_name, labels in probes.items():
    y_sub = labels[sub_idx]
    if y_sub.sum() < 10 or (1 - y_sub).sum() < 10:
        print(f"  {probe_name}: skipped (insufficient labels)")
        continue

    clf = LogisticRegression(max_iter=500, C=1.0, solver="saga", random_state=42)
    scores = cross_val_score(clf, X_sub, y_sub, cv=5, scoring="roc_auc", n_jobs=-1)

    best_single = codon_probe_results[probe_name]["best_auroc"]
    print(f"  {probe_name}:")
    print(f"    Best single latent:    AUROC = {best_single:.4f}")
    print(f"    Logistic regression:   AUROC = {scores.mean():.4f} \u00b1 {scores.std():.4f}")
    gap = scores.mean() - best_single
    if gap < 0.02:
        print("    \u2192 Concept is MONOSEMANTIC (single feature captures it)")
    elif gap < 0.1:
        print("    \u2192 Concept is MOSTLY monosemantic (small multi-feature gain)")
    else:
        print(f"    \u2192 Concept is DISTRIBUTED (large multi-feature gain: +{gap:.3f})")

# Gene-level
print("\nGene-level probes:")
for probe_name, result in gene_probe_results.items():
    if probe_name == "pLI_constrained":
        y = np.array([1.0 if pli_map.get(g, 0) > 0.9 else 0.0 for g in gene_names_unique])
    elif probe_name == "housekeeping_proxy":
        y = housekeeping_labels
    else:
        continue

    if y.sum() < 10 or (1 - y).sum() < 10:
        print(f"  {probe_name}: skipped")
        continue

    clf = LogisticRegression(max_iter=500, C=1.0, solver="saga", random_state=42)
    scores = cross_val_score(clf, gene_mean_acts, y, cv=5, scoring="roc_auc", n_jobs=-1)
    best_single = result["best_auroc"]
    print(f"  {probe_name}:")
    print(f"    Best single latent:    AUROC = {best_single:.4f}")
    print(f"    Logistic regression:   AUROC = {scores.mean():.4f} \u00b1 {scores.std():.4f}")

## Summary

The probing results tell us:

1. **Which biological concepts are monosemantically encoded** — single features that achieve high AUROC are clean detectors. These are the features you'd want to use for steering experiments (e.g., clamping the "rare codon" detector high to generate rare-codon-enriched sequences).

2. **Which concepts are distributed** — if no single feature predicts a label well but logistic regression does, the concept is spread across features. This is still useful (the SAE represents it) but harder to steer with a single knob.

3. **Which concepts the model doesn't represent** — if even logistic regression can't predict a label from SAE activations, the base model likely doesn't encode that property in its hidden states at this layer.

### Adding Custom Probes

To add your own probe, define a binary label array and pass it to `compute_single_latent_aurocs`:

```python
# Example: probe for codons ending in 'A'
my_labels = np.array([1.0 if c.endswith("A") else 0.0 for c in codon_strings])
aurocs, _ = compute_single_latent_aurocs(codon_acts, my_labels, max_features=5000)
best = np.argmax(aurocs)
print(f"Best feature for 'ends with A': {best} (AUROC={aurocs[best]:.4f})")
```

You can also probe for any gene-level property by adding a label per gene in `gene_names_unique` and using `gene_mean_acts`.